# Обучение базовых моделей P3

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 4)

In [ ]:
PATH = Path("..", "data", "processed", "p3_data.csv")
df = pd.read_csv(PATH, sep=";")

DTTM_COLS = [
    'month',
    'first_trx_month_inn',
    'last_active_month_inn',
    'first_month_p1p2',
    'first_month_altpay5',
]

for col in DTTM_COLS:
    df[col] = pd.to_datetime(df[col])

# Приводим bool-колонки к int8, чтобы они корректно обрабатывались в моделях
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].astype('int8')

### Случайный отбор

In [3]:
from acquiring_xsell.metrics import calculate_random_baseline

print("Random Selection P3")

calculate_random_baseline(df=df)

Random Selection P3
K=500  | Precision: 0.01% | Coverage: 1.94% | Expected Conversions: 0.03
K=1000 | Precision: 0.01% | Coverage: 3.87% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Coverage: 5.81% | Expected Conversions: 0.10
K=2000 | Precision: 0.01% | Coverage: 7.74% | Expected Conversions: 0.14
K=2500 | Precision: 0.01% | Coverage: 9.68% | Expected Conversions: 0.17


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `P3`
- Статус `'Active'`, `'Reborn'`
- Релевантные MCC (флаг `is_relevant_mcc_p3`)
- Группа MCC `!= 'Charity'`

In [4]:
from acquiring_xsell.features import add_rule_flag_p3
from acquiring_xsell.metrics import calculate_metrics

df = add_rule_flag_p3(df)

print(f"Business-Rule Selection P3, y_score='turnover'")

calculate_metrics(data=df, y_score='turnover', business_rule_mode=True)

Business-Rule Selection P3, y_score='turnover'
K=500  | Precision: 0.03% | Recall: 7.58% | Expected Conversions: 0.13
K=1000 | Precision: 0.02% | Recall: 10.61% | Expected Conversions: 0.20
K=1500 | Precision: 0.01% | Recall: 10.61% | Expected Conversions: 0.20
K=2000 | Precision: 0.02% | Recall: 28.79% | Expected Conversions: 0.47
K=2500 | Precision: 0.02% | Recall: 31.82% | Expected Conversions: 0.53


Попробуем также как с `P1P2` пересмотреть метрику ранжирования

In [ ]:
numeric_features = [col for col in df.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag')]

print(f"Business-Rule Selection P3\n")

for ascending in (True, False):
    for feature in numeric_features:
        print(f"y_score='{feature}', ascending={ascending}")
        calculate_metrics(data=df, y_score=feature, ascending=ascending, business_rule_mode=True)
        print()

Business-Rule Selection P3

y_score='lifetime_month_streak_inn', ascending=True
K=500  | Precision: 0.01% | Recall: 2.27% | Expected Conversions: 0.07
K=1000 | Precision: 0.01% | Recall: 2.27% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Recall: 4.55% | Expected Conversions: 0.13
K=2000 | Precision: 0.02% | Recall: 12.12% | Expected Conversions: 0.33
K=2500 | Precision: 0.02% | Recall: 25.76% | Expected Conversions: 0.47

y_score='altpay1_turnover', ascending=True
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.01% | Recall: 1.52% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Recall: 6.06% | Expected Conversions: 0.13
K=2000 | Precision: 0.01% | Recall: 12.12% | Expected Conversions: 0.27
K=2500 | Precision: 0.01% | Recall: 14.39% | Expected Conversions: 0.33

y_score='altpay2_turnover', ascending=True
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.02% | Recall: 7.58% | Exp

### Logistic Regression

Подготовим данные для обучения логистической регрессии

In [ ]:
from acquiring_xsell.features import add_months_since_last_active_month
from acquiring_xsell.features import add_months_since_first_trx
from acquiring_xsell.features import add_has_p1p2
from acquiring_xsell.features import add_has_altpay5

df = add_months_since_last_active_month(df)
df = add_months_since_first_trx(df)
df = add_has_p1p2(df)
df = add_has_altpay5(df)

cols_to_drop = [
    'inn',
    "last_active_month_inn",
    "first_trx_month_inn",
    "top_mcc_group_inn",
    "who_is_this_first_inn",
    "first_month_p1p2",
    "first_month_altpay5",
]

df = df.drop(columns=cols_to_drop)

In [ ]:
train = df_p3_preprocessed.loc[df_p3_preprocessed['month'] < '2025-12-01'] # 80%
test = df_p3_preprocessed.loc[df_p3_preprocessed['month'] >= '2025-12-01'] # 20%

X_train = train.drop(columns=['target', 'month'])
y_train = train['target']

X_test = test.drop(columns=['target', 'month'])
y_test = test['target']

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100_000)

lr.fit(X_train, y_train)

test_with_lr_proba = test.copy()
train_with_lr_proba = train.copy()

train_with_lr_proba['lr_proba'] = lr.predict_proba(X_train)[:, 1]
test_with_lr_proba['lr_proba'] = lr.predict_proba(X_test)[:, 1]

In [ ]:
print(f"Logistic Regression P3, y_score='lr_proba'\n")

print(f"Train:")
calculate_metrics(data=train_with_lr_proba, y_score='lr_proba')
print()

print(f"Test:")
calculate_metrics(data=test_with_lr_proba, y_score='lr_proba')
print()

Logistic Regression P3, y_score='lr_proba'

Train:
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1500 | Precision: 0.01% | Recall: 3.70% | Expected Conversions: 0.08
K=2000 | Precision: 0.01% | Recall: 5.56% | Expected Conversions: 0.17
K=2500 | Precision: 0.01% | Recall: 10.19% | Expected Conversions: 0.33

Test:
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1500 | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=2000 | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=2500 | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00



In [ ]:
df_p3_preprocessed.groupby('month').target.sum()

month
2024-12-01    3
2025-01-01    6
2025-02-01    3
2025-03-01    2
2025-04-01    3
2025-05-01    0
2025-06-01    4
2025-07-01    1
2025-08-01    0
2025-09-01    1
2025-10-01    0
2025-11-01    1
2025-12-01    1
2026-01-01    1
2026-02-01    0
Name: target, dtype: int64